# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
!git clone https://github.com/ainasarfaraz343-a11y/flyrank-internship.git
%cd flyrank-internship

import pandas as pd
import numpy as np
import os

np.random.seed(42)

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_stale'] = (df['days_since_last_update'] >= 90).astype(int)
df['has_volume'] = (df['impressions_90d'] >= 300).astype(int)
df['score'] = df['is_stale'] * df['has_volume'] * df['impressions_90d']

def reason_code(row):
    if row['is_stale'] and row['has_volume']:
        return 'STALE_HIGH_VOLUME'
    elif row['is_stale'] and not row['has_volume']:
        return 'STALE_LOW_VOLUME'
    else:
        return 'FRESH_NO_ACTION'

df['reason_code'] = df.apply(reason_code, axis=1)
df['action'] = df['reason_code'].apply(lambda r: 'REVIEW_REFRESH' if r == 'STALE_HIGH_VOLUME' else 'NO_ACTION')

queue = df.sort_values('score', ascending=False)
print(queue[['content_id','action','reason_code','score']].head(10))

Cloning into 'flyrank-internship'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 148 (delta 53), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (148/148), 1.88 MiB | 15.65 MiB/s, done.
Resolving deltas: 100% (53/53), done.
/content/flyrank-internship/flyrank-internship
                 content_id          action        reason_code   score
6653   content_5fe46e04994d  REVIEW_REFRESH  STALE_HIGH_VOLUME  517715
29400  content_2dba2b1f9536  REVIEW_REFRESH  STALE_HIGH_VOLUME  443434
13537  content_2c2606c5d176  REVIEW_REFRESH  STALE_HIGH_VOLUME  347399
26531  content_cb112fce36be  REVIEW_REFRESH  STALE_HIGH_VOLUME  309910
21565  content_9532f197bbc8  REVIEW_REFRESH  STALE_HIGH_VOLUME  309192
3394   content_36ff89c8214e  REVIEW_REFRESH  STALE_HIGH_VOLUME  295097
26798  content_b28d1efd668f  REVIEW_REFRESH  STALE_HIGH_VOLUME  286608
23767  content_813e88069237  RE

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The playbook is built on the Week-4 rule, tested and audited through Weeks
5-6: `STALE_HIGH_VOLUME` → `REVIEW_REFRESH`; everything else → `NO_ACTION`.
The reason code is the whole justification a reviewer needs — no black box.

In [5]:
action_summary = queue['action'].value_counts()
reason_summary = queue['reason_code'].value_counts()
print("Action counts:")
print(action_summary)
print("\nReason code counts:")
print(reason_summary)
print(f"\nTop 10 flagged for review:")
print(queue[['content_id', 'action', 'reason_code', 'days_since_last_update',
             'impressions_90d', 'avg_position', 'ctr']].head(10))


Action counts:
action
NO_ACTION         22766
REVIEW_REFRESH     7234
Name: count, dtype: int64

Reason code counts:
reason_code
FRESH_NO_ACTION      20655
STALE_HIGH_VOLUME     7234
STALE_LOW_VOLUME      2111
Name: count, dtype: int64

Top 10 flagged for review:
                 content_id          action        reason_code  \
6653   content_5fe46e04994d  REVIEW_REFRESH  STALE_HIGH_VOLUME   
29400  content_2dba2b1f9536  REVIEW_REFRESH  STALE_HIGH_VOLUME   
13537  content_2c2606c5d176  REVIEW_REFRESH  STALE_HIGH_VOLUME   
26531  content_cb112fce36be  REVIEW_REFRESH  STALE_HIGH_VOLUME   
21565  content_9532f197bbc8  REVIEW_REFRESH  STALE_HIGH_VOLUME   
3394   content_36ff89c8214e  REVIEW_REFRESH  STALE_HIGH_VOLUME   
26798  content_b28d1efd668f  REVIEW_REFRESH  STALE_HIGH_VOLUME   
23767  content_813e88069237  REVIEW_REFRESH  STALE_HIGH_VOLUME   
26255  content_c21024970297  REVIEW_REFRESH  STALE_HIGH_VOLUME   
7445   content_c8e9d6ab9013  REVIEW_REFRESH  STALE_HIGH_VOLUME   

       da

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use:** This playbook is decision-support for prioritizing which
pages a content team reviews first. On the measured test split, the flagged
pages showed a directional improvement over random selection at precision@50
— useful for ordering a review queue, not for skipping the review itself.

**What it is NOT:**
- Not a prediction that any specific page will decline or recover — the
  model's overall discrimination (ROC-AUC ≈ 0.56, from Week 6) is close to
  chance, so individual-page confidence is low even where the ranking is
  useful in aggregate.
- Not causal — nothing here says refreshing a page *will* fix its
  performance, only that certain pages look worth a human look first.
- Not validated on data outside this 30K-row starter set, or on clients this
  model hasn't seen (Week 6 showed a measurable gap between random and
  client-grouped splits).
- Not a substitute for checking *why* a page is flagged — Week 4's top-10
  review found several flags driven by data artifacts (e.g. zero CTR that
  may be a tracking issue, not staleness).

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Always human-reviewed before action:**
- Any `REVIEW_REFRESH` flag, before committing writer/editor time
- Pages with `ctr == 0` despite real impressions — could be a tracking bug
  or broken link, not a content problem (seen in Week-4's top-10 review)
- Pages already strong on position AND CTR that still got flagged — the
  rule gates on staleness+volume only, so it can flag pages that don't
  actually need work (Week-4 found 3 such weak picks in the top 10)

**No-go — do NOT automate:**
- Auto-publishing any content change based on the score alone
- Treating the score as a performance guarantee to clients
- Using the score to justify removing/deprioritizing a page without a
  human confirming the reason code still applies
- Applying this playbook to a client type not represented in the training
  data without re-checking the signal tables first

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Re-check the rule/model if:**
- The base rate of decline shifts meaningfully from the observed ~54-62%
  range (a sign the underlying population or measurement changed)
- A new content_type or client segment appears with a different missingness
  pattern than what Section-2-style checks assumed
- Precision@50 on a fresh sample drops noticeably below the ~0.44 observed
  in Week 6's honest-split evaluation

**Light monitoring cadence (practical, not production-grade):**
- Re-run the signal bucket tables (Week 4/ML-06 style) monthly or when a
  new data pull arrives
- Spot-check 5-10 flagged pages by hand each cycle, the way the Week-4
  top-10 review did

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Exporting the ranked queue (regenerated each run, stays out of git) and a
metrics JSON (the receipts — committed, since it's small and reproducible).

In [6]:
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

metrics = {
    'total_pages': int(len(df)),
    'flagged_review_refresh': int((df['action'] == 'REVIEW_REFRESH').sum()),
    'reason_code_counts': df['reason_code'].value_counts().to_dict(),
    'note': 'Ranked queue built on Week-4 rule (is_stale x has_volume x impressions_90d), validated Week 5-6.'
}

import json
with open('work/outputs/w07_playbook_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("Exported queue and metrics.")
print(metrics)


Exported queue and metrics.
{'total_pages': 30000, 'flagged_review_refresh': 7234, 'reason_code_counts': {'FRESH_NO_ACTION': 20655, 'STALE_HIGH_VOLUME': 7234, 'STALE_LOW_VOLUME': 2111}, 'note': 'Ranked queue built on Week-4 rule (is_stale x has_volume x impressions_90d), validated Week 5-6.'}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.